# 05 · Agentic loop (Strands: Bedrock Claude + SLM tool) — 도메인 QA / instruction

**TL;DR** — Strands Agent를 구성해 reasoning은 Bedrock Claude가 맡고, 도메인 특화 작업은 파인튜닝한 SLM endpoint를 tool로 호출해 처리합니다.

**Why** — 파인튜닝한 SLM은 특정 작업을 빠르게 처리하는 전문가이고, Claude는 범용 추론과 오케스트레이션을 담당하는 역할로, 각자의 강점에 맞게 업무를 분담합니다.

**기존 Pain Point** — 범용 LLM 하나로 모든 작업을 처리하면 비용이 크고 지연도 큽니다. 특화 SLM을 tool로 연결해 반복 작업을 넘김으로써 비용과 지연을 함께 낮춥니다.

> 🔴 실제 실행 시 AWS 자격증명·GPU·엔드포인트 과금이 발생합니다. 먼저 `DRY_RUN=1`로 파이프라인을 검증하세요.

## 0. 응답 언어 선택
아래 셀의 `LANG`으로 **에이전트가 사용자에게 답하는 언어**를 고릅니다. 프롬프트 자체는 영어로 두고 `' Reply in {LANG}.'` 한 문장만 덧붙이는 방식이라, **번역본을 따로 관리할 필요가 없습니다** — `'Japanese'` 처럼 아무 언어나 넣어도 동작합니다.

🔴 **SLM tool의 출력은 바뀌지 않습니다.** SLM은 학습된 대로 동작하므로(이 kit의 시드는 영어 기반) tool이 돌려주는 값은 그대로이고, 달라지는 것은 **Claude가 그 결과를 설명·요약하는 언어**입니다. 즉 `LANG='Korean'`은 "작업은 영어로 처리하고 설명만 한국어로 해 줘"에 해당합니다.

In [ ]:
# Agent response language. Add any language you want — the prompts stay in English and
# only this instruction is appended, so no translated copies to maintain.
LANG = 'Korean'          # e.g. 'Korean' | 'English' | 'Japanese'
REPLY_IN = f' Reply in {LANG}.'
print('agent replies in:', LANG)

In [ ]:
# Strands 설치 (uv 우선, pip 폴백 — 00_setup과 동일 관용구)
import shutil, subprocess, sys
_pkgs = ['strands-agents>=1.48.0', 'strands-agents-tools']
if shutil.which('uv'):
    subprocess.run(['uv', 'pip', 'install', '--python', sys.executable, '-q', *_pkgs], check=True)
else:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *_pkgs], check=True)
print('installed:', _pkgs)

In [ ]:
import os, sys
# 리포 루트를 path에 추가해 common/ 와 트랙 로컬 모듈을 import
REPO = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.insert(0, REPO)
sys.path.insert(0, os.getcwd())

In [ ]:
import importlib
from common import config; importlib.reload(config)
# 트랙 전용 키 우선 — 전역 endpoint_name 은 다른 트랙이 덮어씁니다.
%store -r ep_domain_qa
%store -r endpoint_name
endpoint_name = globals().get('ep_domain_qa') or globals().get('endpoint_name')
assert endpoint_name, 'endpoint_name 이 없습니다 — 03의 배포 셀을 먼저 실행하세요.'
print('사용할 endpoint:', endpoint_name)
print('endpoint:', endpoint_name)
assert config.BEDROCK_CLAUDE_MODEL_ID, 'BEDROCK_CLAUDE_MODEL_ID env 필요'

## 1. SLM endpoint를 tool로 래핑 (🔴 sagemaker-runtime)
에이전트가 SLM을 호출할 수 있으려면 endpoint 호출 로직을 Strands `@tool` 함수로 감싸야 합니다. tool의 docstring은 에이전트가 이 tool을 언제 사용할지 판단하는 근거가 되므로 역할을 명확히 기술합니다.
🔴 **`messages`로 보냅니다** — vLLM/SGLang/LMI는 OpenAI 호환 서버라 **chat template을 서버가 적용**합니다. 로컬에서 토크나이저로 렌더한 raw 문자열(`{inputs: ...}`)을 보내면 다음 에러가 납니다(실측):
```
Could not find a handler for the request. Expected one of:
  ['ChatCompletionRequest', 'CompletionRequest']
```
그래서 이 tool은 `invoke_sagemaker_chat`을 씁니다 — 토크나이저·transformers 의존도 필요 없습니다.

In [ ]:
from strands import Agent, tool
from common import aws_utils, gemma_format as gf
import importlib, track_data as td; importlib.reload(td)

@tool
def answer_domain_question(text: str) -> str:
    """Answer a domain question (optionally grounded in provided context) using the fine-tuned Gemma SLM."""
    # 🔴 messages 그대로 전송 → 서버(vLLM/SGLang/LMI)가 chat template을 적용합니다.
    msgs = gf.build_inference_messages(text, system_content=td.SYSTEM_PROMPT)
    return aws_utils.invoke_sagemaker_chat(endpoint_name, msgs,
                                          region=config.AWS_REGION,
                                          max_tokens=512, temperature=0.1)

## 2. Bedrock Claude를 reasoning 모델로 Agent 구성 (모델 ID는 env)
Strands는 Bedrock을 기본 프로바이더로 지원하므로, `BedrockModel`에 Claude 모델 ID를 지정해 reasoning 엔진으로 삼습니다. 앞에서 래핑한 SLM tool을 함께 등록하면, 에이전트는 system prompt에 따라 스스로 추론하다가 필요한 시점에 SLM tool을 호출합니다. 모델 ID 같은 값은 코드에 박지 않고 env에서 읽어 옵니다.

In [ ]:
from strands.models import BedrockModel
bedrock_model = BedrockModel(model_id=config.BEDROCK_CLAUDE_MODEL_ID,
                             region_name=config.BEDROCK_REGION)

# The prompt stays in English; REPLY_IN (from the language cell) appends the language
# instruction. Tool output is unaffected — only the agent's own wording changes.
SYSTEM_PROMPT = "You orchestrate. For domain questions, call answer_domain_question, then verify the answer is grounded and add citations if context was provided." + REPLY_IN
agent = Agent(model=bedrock_model,
              tools=[answer_domain_question],
              system_prompt=SYSTEM_PROMPT)
print('system_prompt:', SYSTEM_PROMPT)

### (대안) LiteLLM 프로바이더로 모델 통일
여러 프로바이더의 모델을 동일한 방식으로 다루고 싶다면, Strands의 LiteLLM 프로바이더를 통해 Bedrock 모델을 지정할 수도 있습니다. 아래는 참고용 예시이므로 주석 처리해 두었습니다.

In [ ]:
# %pip install -q 'strands-agents[litellm]'
# from strands.models.litellm import LiteLLMModel
# lm = LiteLLMModel(model_id=f'bedrock/{config.BEDROCK_CLAUDE_MODEL_ID}', params={'max_tokens': 1024})
# agent = Agent(model=lm, tools=[answer_domain_question])

## 3. 에이전트 스모크 (최소 호출 — endpoint + Bedrock 이중 과금 주의)
구성한 에이전트에 예시 입력을 하나 넣어 reasoning → tool 호출 → 응답으로 이어지는 루프가 정상 동작하는지 확인합니다. 이 한 번의 호출에도 Bedrock reasoning과 SLM endpoint가 함께 과금되므로, 검증에는 최소한의 호출만 사용합니다.

In [ ]:
SMOKE_USER = "Using the SLM, answer: 'What is the capital of Australia and why is it not Sydney?'" + REPLY_IN
print('--- user ---\n', SMOKE_USER, '\n')
result = agent(SMOKE_USER)
print(result)

✅ 로컬 agentic 루프가 동작합니다. 프로덕션 배포는 **06_agentcore_deploy.ipynb**로 이어지며, 실습을 마치려면 99_cleanup으로 정리하세요.